In [1]:
library(MLmetrics)
library(randomForest)
library(dplyr) 

ConfusionMatrix <- function(y_pred, y_true) {
  Confusion_Mat <- table(y_true, y_pred)
  return(Confusion_Mat)
}
 
ConfusionDF <- function(y_pred, y_true) {
  Confusion_DF <- transform(as.data.frame(ConfusionMatrix(y_pred, y_true)),
                            y_true = as.character(y_true),
                            y_pred = as.character(y_pred),
                            Freq = as.integer(Freq))
  return(Confusion_DF)
}
 
Precision_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FP <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # it may happen that a label is never predicted (missing from y_pred) but exists in y_true
    # in this case ConfusionDF will not have these lines and thus the simplified code crashes
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]))
   
    # workaround:
    # i don't want to change ConfusionDF since i don't know if the current behaviour is a feature or a bug.
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
   
    tmp <- Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]
    FP[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Precision_micro <- sum(TP) / (sum(TP) + sum(FP))
  return(Precision_micro)
}
 
Recall_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FN <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # short version, comment out due to bug or feature of Confusion_DF
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]))
   
    # workaround:
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
 
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]
    FN[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Recall_micro <- sum(TP) / (sum(TP) + sum(FN))
  return(Recall_micro)
}
 
F1_Score_micro <- function(y_true, y_pred, labels = NULL) {
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred)) # possible problems if labels are missing from y_*
  Precision <- Precision_micro(y_true, y_pred, labels)
  Recall <- Recall_micro(y_true, y_pred, labels)
  F1_Score_micro <- 2 * (Precision * Recall) / (Precision + Recall)
  return(F1_Score_micro)
}

Warning message:
"le package 'MLmetrics' a été compilé avec la version R 4.2.3"

Attachement du package : 'MLmetrics'


L'objet suivant est masqué depuis 'package:base':

    Recall


Warning message:
"le package 'randomForest' a été compilé avec la version R 4.2.3"
randomForest 4.7-1.1

Type rfNews() to see new features/changes/bug fixes.

Warning message:
"le package 'dplyr' a été compilé avec la version R 4.2.3"

Attachement du package : 'dplyr'


L'objet suivant est masqué depuis 'package:randomForest':

    combine


Les objets suivants sont masqués depuis 'package:stats':

    filter, lag


Les objets suivants sont masqués depuis 'package:base':

    intersect, setdiff, setequal, union




In [2]:
datam<-read.csv("data_target_encoding.csv",stringsAsFactors = T)

In [3]:
set.seed(2)
k = 10

accuracy_vec <- array(0,k)

n_trees <- c(10,20,100)
nfeat <- ncol(datam)-1 #I remove 2 to remove building_id and damage_grade
m_tries <- c(floor(0.5*sqrt(nfeat)),floor(sqrt(nfeat)),floor(2*sqrt(nfeat)))
accuracy_vec <- data.frame(matrix(ncol = 4, nrow = 0))
colnames(accuracy_vec)<-c('fold','n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
target_variable <- match('damage_grade', colnames(datam)) #we will remove id.
pb <- txtProgressBar(min = 0, max = length(n_trees), style = 3)

# 1. Shuffle the dataset randomly.
datam_idx <- sample(1:nrow(datam))

# 2. Split the dataset into k groups
max <- ceiling(nrow(datam)/k)
splits <- split(datam_idx, ceiling(seq_along(datam_idx)/max))

pb <- txtProgressBar(min = 0, max = k*length(n_trees)*length(m_tries), style = 3)
counter <-0

# 3. For each unique group:
for (i in 1:k){
    for (j in (n_trees)){
        for (l in m_tries){
            #3.1 Take the group as a hold out or test data set
            test_data <- datam[splits[[i]],]


            #3.2 Take the remaining groups as a training data set
            train_data <- datam[-splits[[i]],]   

            model <- randomForest(x=train_data[,-c(target_variable)],
                                y=as.factor(train_data[,c(target_variable)]),
                                ntree=j,mtry=l,keep.forest=TRUE,importance=TRUE)
            yhat<-predict(model,test_data[,-c(target_variable)])
            accuracy_vec[nrow(accuracy_vec)+1,]<-c(i,j,l,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
            counter<-counter+1
            setTxtProgressBar(pb, counter)
            print(paste("Best F1 Score - ",i, 'fold',j,'trees',l,'mtry',':',accuracy_vec[nrow(accuracy_vec),4],))
            rm('model')
        }
    }
}

#4. Summarize the accuracy of the model using the sample of model evaluation scores
meanF1 <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(meanF1)<-c('n_trees','mtry','mean F1')
for (i in n_trees){
    for (j in m_tries){
            meanF1[nrow(meanF1)+1,]<-c(i,j,mean(filter(accuracy_vec,(n_trees==i & mtry==j))$F1))
    }
}
accuracy_vec
meanF1

  |=                                                                     |   1%[1] "Best F1 Score -  1 fold 10 trees 3 mtry : 0.752542112735505 1"
  |==                                                                    |   2%[1] "Best F1 Score -  1 fold 10 trees 6 mtry : 0.746786385787192 1"
  |==                                                                    |   3%[1] "Best F1 Score -  1 fold 10 trees 12 mtry : 0.731591266643644 1"
  |===                                                                   |   4%[1] "Best F1 Score -  1 fold 20 trees 3 mtry : 0.753808372664134 1"
  |====                                                                  |   6%[1] "Best F1 Score -  1 fold 20 trees 6 mtry : 0.752887456352404 1"
  |=====                                                                 |   7%[1] "Best F1 Score -  1 fold 20 trees 12 mtry : 0.74064694370899 1"
  |=====                                                                 |   8%[1] "Best F1 Score -  1 fold 100 trees

fold,mean_F1
<dbl>,<dbl>
1,0.7489736
2,0.7488883
3,0.7464666
4,0.7458612
5,0.7503038
6,0.7475837
7,0.7503080
8,0.7504828
9,0.7513910


In [31]:
meanF1 <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(meanF1)<-c('n_trees','mtry','mean F1')
for (i in n_trees){
    for (j in m_tries){
            meanF1[nrow(meanF1)+1,]<-c(i,j,mean(filter(accuracy_vec,(n_trees==i & mtry==j))$F1))
    }
}
meanF1

,n_trees,mtry,mean F1
,<dbl>,<dbl>,<dbl>
1,10,3,0.7512787
2,10,6,0.7450010
3,10,12,0.7338959
4,20,3,0.7552196
5,20,6,0.7512903
6,20,12,0.7410946
7,100,3,0.7580324
8,100,6,0.7569426
9,100,12,0.7474876


In [3]:
set.seed(2)
k = 10

accuracy_vec <- array(0,k)

n_trees <- c(200,500)
nfeat <- ncol(datam)-1 #I remove 2 to remove building_id and damage_grade
m_tries <- c(floor(0.5*sqrt(nfeat)),floor(sqrt(nfeat)))
accuracy_vec <- data.frame(matrix(ncol = 4, nrow = 0))
colnames(accuracy_vec)<-c('fold','n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
target_variable <- match('damage_grade', colnames(datam)) #we will remove id.
pb <- txtProgressBar(min = 0, max = length(n_trees), style = 3)

# 1. Shuffle the dataset randomly.
datam_idx <- sample(1:nrow(datam))

# 2. Split the dataset into k groups
max <- ceiling(nrow(datam)/k)
splits <- split(datam_idx, ceiling(seq_along(datam_idx)/max))

pb <- txtProgressBar(min = 0, max = k*length(n_trees)*length(m_tries), style = 3)
counter <-0

# 3. For each unique group:
for (i in 1:k){
    for (j in (n_trees)){
        for (l in m_tries){
            #3.1 Take the group as a hold out or test data set
            test_data <- datam[splits[[i]],]


            #3.2 Take the remaining groups as a training data set
            train_data <- datam[-splits[[i]],]   

            model <- randomForest(x=train_data[,-c(target_variable)],
                                y=as.factor(train_data[,c(target_variable)]),
                                ntree=j,mtry=l,keep.forest=TRUE,importance=TRUE)
            yhat<-predict(model,test_data[,-c(target_variable)])
            accuracy_vec[nrow(accuracy_vec)+1,]<-c(i,j,l,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
            counter<-counter+1
            setTxtProgressBar(pb, counter)
            print(paste("Best F1 Score - ",i, 'fold',j,'trees',l,'mtry',':',accuracy_vec[nrow(accuracy_vec),4]))
            rm('model')
        }
    }
}

#4. Summarize the accuracy of the model using the sample of model evaluation scores
meanF1 <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(meanF1)<-c('n_trees','mtry','mean F1')
for (i in n_trees){
    for (j in m_tries){
            meanF1[nrow(meanF1)+1,]<-c(i,j,mean(filter(accuracy_vec,(n_trees==i & mtry==j))$F1))
    }
}
accuracy_vec
meanF1

  |==                                                                    |   2%[1] "Best F1 Score -  1 fold 200 trees 3 mtry : 0.757683895475999 1"
  |====                                                                  |   5%[1] "Best F1 Score -  1 fold 200 trees 6 mtry : 0.758681554813706 1"
  |=====                                                                 |   8%[1] "Best F1 Score -  1 fold 500 trees 3 mtry : 0.758950155404628 1"
  |=======                                                               |  10%[1] "Best F1 Score -  1 fold 500 trees 6 mtry : 0.759103641456583 1"
  |=========                                                             |  12%[1] "Best F1 Score -  2 fold 200 trees 3 mtry : 0.75979432869038 1"
  |==========                                                            |  15%[1] "Best F1 Score -  2 fold 200 trees 6 mtry : 0.758374582709796 1"
  |============                                                          |  18%[1] "Best F1 Score -  2 fold 500 t

,n_trees,mtry,mean F1
,<dbl>,<dbl>,<dbl>
1,200,3,0.7588344
2,200,6,0.7577638
3,500,3,0.7592564
4,500,6,0.7579940
